# XEUSS GIWAXS Data Loading and Reciprocal-Space Mapping

This notebook processes one Xeuss GISAXS detector frame (`.edf`) into calibrated reciprocal-space maps and optional 1D/chi-cut outputs.

Run order:

1. Set paths and analysis parameters.
2. Load the detector frame and read the Xeuss header metadata.
3. Build a `.poni` calibration file from the detector header.
4. Transform the raw image to $(q_r, q_z)$ reciprocal space.
5. Apply the GISAXS missing-wedge mask and save maps/arrays.
6. Optionally export rotational averages, chi cuts, indexed overlays, TIF copies, and simulation overlays.

Edit only the **Settings** cell for normal use. Optional sections are controlled by flags and path variables in that cell.

## 0. Imports

The notebook uses `fabio` for detector files, `pyFAI` for geometry/calibration, and `pygix` for GIWAXS reciprocal-space transforms.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch, Annulus

#%matplotlib inline
%matplotlib widget 
# if ipympl is installed and you want interactive zoom/pan inside JupyterLab

import fabio
import pyFAI
import pygix

mpl.rcParams["text.usetex"] = False

## 1. Settings

Change the paths and analysis parameters here, then run the notebook from top to bottom.

In [ ]:
# --- Required input/output paths ---------------------------------------------
data_folder = Path("SOME_DATA_FOLDER")
data_filename = "FILENAME.edf"
sample_name = "SAMPLE_NAME"
output_folder = data_folder / "output"

# --- Reciprocal-space transform ----------------------------------------------
npt = (1000, 500)                 # optional grid resolution (qr, qz); see transform cell
sample_orientation = 3            # pygix frame convention; change if axes look flipped
tilt_angle = 0.0                  # sample tilt about the beam (deg)
correct_solid_angle = True

# --- Plot appearance ----------------------------------------------------------
cmap = "viridis"
vmin_percentile = 5.0             # lower colour-scale clip from positive pixels
vmax_percentile = 99.5            # upper colour-scale clip from positive pixels
map_xlim_A = (0, 3)               # q_r limits for publication-style map, A^-1
map_ylim_A = (0, 3)               # q_z limits for publication-style map, A^-1

# --- Optional diagnostics / exports ------------------------------------------
show_full_header = False
save_figures = True
save_arrays = True

# Optional hkl overlay on the 2D reciprocal-space map. Can be generated using the INSIGHT package
hkl_csv_path = Path(rf"SOME_PATH_{sample_name}.csv")
plot_hkl_overlay = True

# Optional rotational-average overlay with simulated/reference data.
simulation_xlsx_path = Path(rf"SOME_PATH_{sample_name}_results.xlsx")
plot_simulation_overlay = True

# Optional EDF-to-TIF conversion. Leave disabled unless you need a raw-image copy.
export_tif_copy = False
edf_to_tif_input = data_folder / "FILENAME.edf"
edf_to_tif_output = data_folder / "FILENAME.tif"

# --- Rotational average settings ---------------------------------------------
azimuth_range = (-90, 0)          # deg; sample-side sector, verify sign for your geometry
npt_q = 1000

# --- Derived paths ------------------------------------------------------------
data_file = data_folder / data_filename
poni_file = data_file.with_suffix(".poni")
output_folder.mkdir(parents=True, exist_ok=True)

print("Data file :", data_file)
print("PONI file :", poni_file)
print("Output dir:", output_folder)

## 2. Helper Functions

These small utilities keep the later analysis cells short and make repeated plot/export behavior consistent.

In [ ]:
def require_header_value(header, *names):
    """Return the first available header value from a list of possible key names."""
    for name in names:
        if name in header:
            return header[name]
    raise KeyError(f"None of these header keys were found: {names}")


def positive_lognorm(values, mask=None, vmin_pct=vmin_percentile, vmax_pct=vmax_percentile):
    """Create a LogNorm from positive finite values, optionally excluding a boolean mask."""
    arr = np.asarray(values)
    valid = np.isfinite(arr) & (arr > 0)
    if mask is not None:
        valid &= ~np.asarray(mask, dtype=bool)
    pos = arr[valid]
    if pos.size == 0:
        raise ValueError("No positive finite values available for logarithmic colour scaling.")
    return mcolors.LogNorm(
        vmin=np.nanpercentile(pos, vmin_pct),
        vmax=np.nanpercentile(pos, vmax_pct),
    )


def save_figure(fig, filename, dpi=150, sample_label=None, **kwargs):
    """Save a figure into output_folder and prefix the sample/frame name."""
    if save_figures:
        filename = Path(filename)
        sample_label = sample_label or data_file.stem
        stem = filename.stem if filename.stem.startswith(f"{sample_label}_") else f"{sample_label}_{filename.stem}"
        path = output_folder / filename.parent / f"{stem}{filename.suffix}"
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=dpi, bbox_inches=kwargs.pop("bbox_inches", "tight"), **kwargs)
        print("Saved:", path)


def save_uint16_tif(array, mask, filename, sample_label=None,
                    vmin_pct=vmin_percentile, vmax_pct=vmax_percentile):
    """Save a masked intensity array as a scaled 16-bit TIFF image."""
    filename = Path(filename)
    sample_label = sample_label or data_file.stem
    stem = filename.stem if filename.stem.startswith(f"{sample_label}_") else f"{sample_label}_{filename.stem}"
    path = output_folder / filename.parent / f"{stem}.tif"
    path.parent.mkdir(parents=True, exist_ok=True)

    arr = np.asarray(array, dtype=float)
    valid = np.isfinite(arr) & (arr > 0) & ~np.asarray(mask, dtype=bool)
    if not np.any(valid):
        raise ValueError("No positive unmasked values available for uint16 TIFF export.")

    vmin = np.nanpercentile(arr[valid], vmin_pct)
    vmax = np.nanpercentile(arr[valid], vmax_pct)
    scaled = np.zeros(arr.shape, dtype=np.uint16)
    scaled_values = np.clip((arr - vmin) / (vmax - vmin), 0, 1)
    scaled[valid] = np.round(scaled_values[valid] * np.iinfo(np.uint16).max).astype(np.uint16)

    try:
        import tifffile
        tifffile.imwrite(path, scaled)
    except ImportError:
        from PIL import Image
        Image.fromarray(scaled).save(path)
    print("Saved:", path)

## 3. Load the Detector Frame

The Xeuss `.edf` header contains the detector geometry and incident angle used in the calibration and GISAXS transform.

In [ ]:
img = fabio.open(str(data_file))
data = img.data.astype(float)
header = img.header

omega = float(require_header_value(header, "om"))

print(f"Incident angle om = {omega:.4f} deg")
print(f"Image shape       = {data.shape}")

if show_full_header:
    for key in sorted(header):
        print(f"{key:>20s} : {header[key]}")

## 4. Build the `.poni` Calibration File

The `.poni` file stores sample-detector distance, beam centre, pixel size, and wavelength. It is created from the detector header so that the transform can be reproduced later.

In [ ]:
ai = pyFAI.AzimuthalIntegrator(
    dist=float(require_header_value(header, "SampleDistance")),
    poni1=float(require_header_value(header, "Center_2")) * float(require_header_value(header, "PSize_2")),
    poni2=float(require_header_value(header, "Center_1")) * float(require_header_value(header, "PSize_1")),
    pixel1=float(require_header_value(header, "PSize_2")),
    pixel2=float(require_header_value(header, "PSize_1")),
    wavelength=float(require_header_value(header, "WaveLength", "Wavelength")),
)
ai.save(str(poni_file))

print("Saved:", poni_file)
print(f"SDD:        {float(require_header_value(header, 'SampleDistance')) * 1000:.1f} mm")
print(f"Wavelength: {float(require_header_value(header, 'WaveLength', 'Wavelength')) * 1e10:.5f} A")
print(
    "Center:     "
    f"({float(require_header_value(header, 'Center_1')):.1f}, "
    f"{float(require_header_value(header, 'Center_2')):.1f}) px"
)

## 5. Raw Detector Image Check

Inspect the raw frame before transforming it. Adjust `x0` and `x1` if a different detector region is more useful.

In [ ]:
x0, x1 = 1000, 2200
region = data[:, x0:x1]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(
    data,
    origin="upper",
    cmap=cmap,
    norm=positive_lognorm(region),
)
ax.set_xlim(x0, x1)
ax.set_xlabel("x / pixel")
ax.set_ylabel("y / pixel")
fig.colorbar(im, ax=ax, label="Intensity (counts)")
fig.tight_layout()
save_figure(fig, f"{data_file.stem}_{sample_name}_Raw_detector_frame.png")
plt.show()

## 6. Transform to Reciprocal Space

`pygix` converts the detector image into $(q_r, q_z)$ coordinates. The map axes are in A$^{-1}$.

In [ ]:
pg = pygix.Transform()
pg.load(str(poni_file))
pg.incident_angle = omega
pg.tilt_angle = tilt_angle
pg.sample_orientation = sample_orientation

# The original workflow used pygix's default reciprocal grid. To force a specific
# output resolution, use: pg.transform_reciprocal(data, npt=npt, unit="A", ...)
intensity, qxy, qz = pg.transform_reciprocal(
    data,
    unit="A",
    correctSolidAngle=correct_solid_angle,
)
intensity_pol, q_abs, chi = pg.transform_polar(data)

print("Reciprocal map shape:", intensity.shape)
print("Polar map shape:     ", intensity_pol.shape)
print(f"q_r range: {np.nanmin(qxy):.4f} to {np.nanmax(qxy):.4f} A^-1")
print(f"q_z range: {np.nanmin(qz):.4f} to {np.nanmax(qz):.4f} A^-1")

## 7. Plot the Unmasked Reciprocal-Space Map

This plot is useful for confirming that the transform and orientation settings are reasonable before applying masks.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
pcm = ax.pcolormesh(
    qxy,
    qz,
    intensity,
    norm=positive_lognorm(intensity),
    cmap=cmap,
    shading="auto",
)
ax.set_xlabel(r"$q_r$ ($\AA^{-1}$)")
ax.set_ylabel(r"$q_z$ ($\AA^{-1}$)")
ax.set_aspect("equal")
fig.colorbar(pcm, ax=ax, label="Intensity (a.u.)")
fig.tight_layout()
save_figure(fig, f"{data_file.stem}_{sample_name}_Intensity_map_uncorrected.png")
plt.show()

## 8. Missing Wedge and Detector Mask

The GISAXS missing wedge is calculated from Ewald-sphere geometry. Detector pixels with negative counts are also flagged as invalid for detector-space operations.

In [ ]:
lam_A = float(require_header_value(header, "WaveLength", "Wavelength")) * 1e10
k = 2 * np.pi / lam_A
alpha_i = np.deg2rad(omega)

QXY, QZ = np.meshgrid(qxy, qz)
sin_af = QZ / k - np.sin(alpha_i)
valid_exit_angle = np.abs(sin_af) <= 1.0
alpha_f = np.arcsin(np.clip(sin_af, -1.0, 1.0))
qr_min = np.abs(k * (np.cos(alpha_f) - np.cos(alpha_i)))

missing_wedge = (np.abs(QXY) < qr_min) | ~valid_exit_angle
mask = data < 0

print(f"Missing-wedge pixels: {missing_wedge.sum()} of {missing_wedge.size}")
print(f"Masked detector pixels: {int(mask.sum())} of {mask.size}")

## 9. Plot the Masked Reciprocal-Space Map

The missing wedge is shown in white. Use this version for most GISAXS map exports.

In [ ]:
intensity_masked = np.ma.masked_where(missing_wedge, intensity)
cmap_masked = plt.get_cmap(cmap).copy()
cmap_masked.set_bad(color="white")

fig, ax = plt.subplots(figsize=(10, 6))
pcm = ax.pcolormesh(
    qxy,
    qz,
    intensity_masked,
    norm=positive_lognorm(intensity, missing_wedge),
    cmap=cmap_masked,
    shading="auto",
)
ax.set_xlabel(r"$q_r$ ($\AA^{-1}$)")
ax.set_ylabel(r"$q_z$ ($\AA^{-1}$)")
ax.set_aspect("equal")
ax.set_xlim(*map_xlim_A)
ax.set_ylim(*map_ylim_A)
fig.colorbar(pcm, ax=ax, label="Intensity (a.u.)")
fig.tight_layout()
save_figure(fig, f"{data_file.stem}_{sample_name}_Intensity_map.png", dpi=500)
save_uint16_tif(intensity, missing_wedge, f"{data_file.stem}_{sample_name}_Intensity_map_uint16.tif")
plt.show()

## 10. Optional Indexed Map Overlay

If `plot_hkl_overlay` is enabled and `hkl_csv_path` exists, this cell overlays indexed peak positions from a CSV with columns `q_r`, `q_z`, and `label`.

In [ ]:
if plot_hkl_overlay and hkl_csv_path.exists():
    hkl = pd.read_csv(hkl_csv_path)
    required = {"q_r", "q_z", "label"}
    missing = required.difference(hkl.columns)
    if missing:
        raise ValueError(f"HKL CSV is missing required columns: {sorted(missing)}")

    fig, ax = plt.subplots(figsize=(10, 6))
    pcm = ax.pcolormesh(
        qxy,
        qz,
        intensity_masked,
        norm=positive_lognorm(intensity, missing_wedge),
        cmap=cmap_masked,
        shading="auto",
    )
    ax.set_xlabel(r"$q_r$ ($\AA^{-1}$)")
    ax.set_ylabel(r"$q_z$ ($\AA^{-1}$)")
    ax.set_aspect("equal")
    ax.set_xlim(*map_xlim_A)
    ax.set_ylim(*map_ylim_A)
    fig.colorbar(pcm, ax=ax, label="Intensity (a.u.)")

    valid_hkl = hkl["q_r"].notna() & hkl["q_z"].notna()
    ax.plot(hkl.loc[valid_hkl, "q_r"], hkl.loc[valid_hkl, "q_z"], "o",
            ms=5, mfc="none", mec="black", mew=0.8, ls="")
    for _, row in hkl.loc[valid_hkl].iterrows():
        label = str(row["label"]).replace("(", "").replace(")", "")
        ax.annotate(
            label,
            (row["q_r"], row["q_z"]),
            textcoords="offset points",
            xytext=(-12, 6),
            va="center",
            fontsize=10,
            color="black",
            weight="bold",
        )

    fig.tight_layout()
    save_figure(fig, f"{data_file.stem}_{sample_name}_Intensity_map_indexed.tiff", dpi=500)
    plt.show()
else:
    print("Skipped indexed overlay. Enable plot_hkl_overlay and provide an existing hkl_csv_path to run it.")

## 11. Save Reciprocal-Space Arrays

The `.npz` file stores the transformed map and masks so the figures can be recreated without rerunning the transform.

In [ ]:
if save_arrays:
    npz_path = output_folder / f"{data_file.stem}_{sample_name}_Intensity_map.npz"
    np.savez(
        npz_path,
        intensity=intensity,
        qxy=qxy,
        qz=qz,
        missing_wedge=missing_wedge,
        mask=mask,
    )
    print("Saved:", npz_path)
else:
    print("Skipped array export because save_arrays is False.")

Reload saved arrays later with:

```python
d = np.load(output_folder / f"{data_file.stem}_{sample_name}_Intensity_map.npz")
intensity = d["intensity"]
qxy = d["qxy"]
qz = d["qz"]
missing_wedge = d["missing_wedge"]
mask = d["mask"]
```

## 12. Rotational Average: I(q)

This section computes and exports a 1D rotational/azimuthal average. `pyFAI` returns `q_nm^-1` here; the map above uses A$^{-1}$.

In [ ]:
azim = ai.integrate1d(
    data,
    npt_q,
    unit="q_nm^-1",
    mask=None,
    azimuth_range=azimuth_range,
)

q_profile = azim.radial
I_profile = azim.intensity

print(
    f"Profile points: {q_profile.size} | "
    f"q range: {q_profile.min():.3f}-{q_profile.max():.3f} nm^-1"
)

profile_df = pd.DataFrame({"q (1/nm)": q_profile, "I (a.u.)": I_profile})
xlsx_path = output_folder / f"{data_file.stem}_{sample_name}_Rotational_average.xlsx"
profile_df.to_excel(xlsx_path, index=False, sheet_name="rotational_average")
print("Saved:", xlsx_path)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(q_profile, I_profile, lw=1.2, color="C9",
        label=rf"RotAve, $\alpha_i$={omega:.3f}$\degree$")
ax.set_yscale("log")
ax.set_xlabel(r"$q$ / nm$^{-1}$")
ax.set_ylabel("Intensity / a.u.")
ax.set_xlim(-0.5, 30)
ax.set_ylim(bottom=np.nanpercentile(I_profile[I_profile > 0], 1))
ax.legend()
fig.tight_layout()
save_figure(fig, f"{data_file.stem}_{sample_name}_Rotational_average.png")
plt.show()

In [ ]:
q_A = q_profile / 10.0
s_nm = q_profile / (2 * np.pi)


def plot_profile(x, filename, xlabel, xlim=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x, I_profile, lw=1.2, color="C9",
            label=rf"RotAve, $\alpha_i$={omega:.3f}$\degree$")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Intensity / a.u.")
    if xlim is not None:
        ax.set_xlim(*xlim)
    ax.set_ylim(bottom=np.nanpercentile(I_profile[I_profile > 0], 1))
    ax.legend()
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()


plot_profile(q_A, f"{data_file.stem}_{sample_name}_Rotational_average_invAngstrom.png",
             r"$q$ / $\AA^{-1}$", xlim=(0.15, 3.0))
plot_profile(s_nm, f"{data_file.stem}_{sample_name}_Rotational_average_invNm.png",
             r"$q/2\pi$ / nm$^{-1}$", xlim=(0.25, 5.0))

## 13. Chi Cuts

Each chi cut integrates over a radial q-band. The ring definitions are in nm$^{-1}$ and are configured in the **Settings** cell.

In [ ]:
# --- Chi-cut settings ---------------------------------------------------------
rings = [
    (10.1, 11.1, "Ring 1"),
    (11.3, 12.5, "Ring 2"),
    (13.0, 14.0, "Ring 3"),
]
chi_npt = 200
chi_range = (-90, 90)             # deg; chi=0 is out-of-plane/surface-normal

ring_params = [
    {
        "label": label,
        "q_min": qmin,
        "q_max": qmax,
        "radial_pos": (qmin + qmax) / 2,
        "radial_width": qmax - qmin,
    }
    for qmin, qmax, label in rings
]

print(f"Incident angle alpha_i = {omega:.4f} deg")
for ring in ring_params:
    print(
        f"{ring['label']:8s} q = {ring['q_min']:.1f}-{ring['q_max']:.1f} nm^-1 "
        f"-> center {ring['radial_pos']:.2f}, width {ring['radial_width']:.2f} nm^-1"
    )

In [ ]:
chi_qr = qxy * 10.0
chi_qz = qz * 10.0
colors_rings = plt.cm.tab10(np.linspace(0, 0.5, len(ring_params)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax2d = axes[0]
pcm = ax2d.pcolormesh(
    chi_qr,
    chi_qz,
    intensity_masked,
    norm=positive_lognorm(intensity, missing_wedge),
    cmap=cmap_masked,
    shading="auto",
)
ax2d.set_xlabel(r"$q_r$ / nm$^{-1}$")
ax2d.set_ylabel(r"$q_z$ / nm$^{-1}$")
ax2d.set_title(f"{data_filename} | chi-cut windows")
ax2d.set_aspect("equal")
ax2d.set_xlim(0, 20)
ax2d.set_ylim(0, 20)
fig.colorbar(pcm, ax=ax2d, label="Intensity (a.u.)")

for ring, color in zip(ring_params, colors_rings):
    for q in (ring["q_min"], ring["q_max"]):
        ax2d.add_patch(plt.Circle((0, 0), q, color=color, fill=False,
                                  linestyle="--", linewidth=1.5))
    ax2d.add_patch(Annulus((0, 0), r=ring["q_max"], width=ring["radial_width"],
                           color=color, alpha=0.25))
ax2d.legend(handles=[
    Patch(color=color, alpha=0.5,
          label=f"{ring['label']}: {ring['q_min']}-{ring['q_max']} nm$^{{-1}}$")
    for ring, color in zip(ring_params, colors_rings)
], loc="upper right", fontsize=8)

ax1d = axes[1]
ax1d.plot(q_profile, I_profile, color="C9",
          label=rf"RotAve | $\alpha_i$={omega:.3f}$\degree$")
for ring, color in zip(ring_params, colors_rings):
    ax1d.axvspan(ring["q_min"], ring["q_max"], color=color, alpha=0.25,
                 label=f"{ring['label']}: {ring['q_min']}-{ring['q_max']} nm$^{{-1}}$")
    ax1d.axvline(ring["q_min"], color=color, linestyle="--", linewidth=1)
    ax1d.axvline(ring["q_max"], color=color, linestyle="--", linewidth=1)
ax1d.set_xlabel(r"$q$ / nm$^{-1}$")
ax1d.set_ylabel("Intensity / a.u.")
ax1d.set_yscale("log")
ax1d.set_title("Rotational average with chi-cut windows")
ax1d.set_xlim(1.0, 25)
ax1d.legend(fontsize=8)

fig.suptitle(rf"chi-cut sanity check | {data_filename} | $\alpha_i$={omega:.4f}$\degree$", fontsize=11)
fig.tight_layout()
save_figure(fig, f"{data_file.stem}_{sample_name}_Chi_cuts_sanitycheck.png")
plt.show()

In [ ]:
chi_results = []
for ring in ring_params:
    pg._chimask = None  # pygix caches this mask; reset so each ring gets its own mask.
    I_chi, chi_axis = pg.profile_chi(
        data,
        npt=chi_npt,
        radial_pos=ring["radial_pos"],
        radial_width=ring["radial_width"],
        chi_range=chi_range,
        unit="q_nm^-1",
        mask=None,
        correctSolidAngle=correct_solid_angle,
    )
    chi_results.append({
        "label": ring["label"],
        "q_min": ring["q_min"],
        "q_max": ring["q_max"],
        "chi_axis": chi_axis,
        "intensity": I_chi,
    })
    print(
        f"{ring['label']:8s} {len(chi_axis)} points | "
        f"chi = {chi_axis[0]:.1f} ... {chi_axis[-1]:.1f} deg"
    )

In [ ]:
chi_export = f"{data_file.stem}_{sample_name}"
chi_folder = output_folder / "Chi_cuts"
chi_folder.mkdir(parents=True, exist_ok=True)

n_rings = len(chi_results)
fig, axes = plt.subplots(1, n_rings, figsize=(7 * n_rings, 5), sharey=False)
if n_rings == 1:
    axes = [axes]
colors_rings = plt.cm.tab10(np.linspace(0, 0.5, n_rings))

for ax, result, color in zip(axes, chi_results, colors_rings):
    ax.plot(
        result["chi_axis"],
        result["intensity"],
        color=color,
        label=(
            rf"{result['label']} | $q$ = {result['q_min']:.1f}-{result['q_max']:.1f} nm$^{{-1}}$ "
            rf"| $\alpha_i$ = {omega:.3f}$\degree$"
        ),
    )
    for marker in (-90, 0, 90):
        ax.axvline(marker, color="grey", linestyle=":" , linewidth=0.8)
    ax.set_xlabel(r"$\chi$ / deg")
    ax.set_ylabel("Intensity / a.u.")
    ax.set_title(result["label"])
    ax.legend(fontsize=8)

fig.suptitle(
    f"chi-cuts | {data_filename} | alpha_i={omega:.4f} deg\n"
    "chi = 0 deg -> face-on | chi = +/-90 deg -> edge-on",
    fontsize=11,
)
fig.tight_layout()
save_figure(fig, Path("Chi_cuts") / f"{data_file.stem}_{sample_name}_ChiCuts.png")

for result in chi_results:
    safe_label = result["label"].replace(" ", "_").replace("?", "-")
    output_path = chi_folder / f"{chi_export}_ChiCut_{safe_label}.xy"
    np.savetxt(
        output_path,
        np.column_stack((result["chi_axis"], result["intensity"])),
        header="chi_deg intensity",
        comments="",
    )
    print("Saved:", output_path)

plt.show()

## 14. Optional EDF to TIF Conversion

Enable `export_tif_copy` in the **Settings** cell when you need a TIF copy of a raw EDF frame.

In [ ]:
if export_tif_copy:
    edf_image = fabio.open(str(edf_to_tif_input))
    edf_image.convert("TIF").save(str(edf_to_tif_output))
    print(f"Converted {edf_to_tif_input} -> {edf_to_tif_output}")
else:
    print("Skipped EDF-to-TIF conversion because export_tif_copy is False.")

## 15. Optional Simulation and Experimental 1D-RotAverage Overlay

If `plot_simulation_overlay` is enabled and `simulation_xlsx_path` exists, this compares the experimental rotational average with simulated/reference data. The workbook is expected to contain sheets named `reflexes` and `radial_average`.

In [ ]:
if plot_simulation_overlay and simulation_xlsx_path.exists():
    reflexes = pd.read_excel(simulation_xlsx_path, sheet_name="reflexes")
    radial = pd.read_excel(simulation_xlsx_path, sheet_name="radial_average")

    reflex_q = reflexes["q (A^-1)"].to_numpy(float)
    reflex_hkl = reflexes["hkl"].astype(str)
    rad_q = radial["q (A^-1)"].to_numpy(float)
    rad_I = radial["I_avg"].to_numpy(float)

    fig, ax = plt.subplots(figsize=(8, 5))
    xlim = (0.15, 3.0)

    ax.plot(q_A, I_profile, lw=1.2, color="C9",
            label=rf"RotAve, $\alpha_i$={omega:.3f}$\degree$")
    ax.set_yscale("log")
    ax.set_xlabel(r"$q$ / $\AA^{-1}$")
    ax.set_ylabel("RotAve intensity / a.u. (log)")
    ax.set_xlim(*xlim)
    ax.set_ylim(bottom=np.nanpercentile(I_profile[I_profile > 0], 1), top=1e2)

    ax2 = ax.twinx()
    ax2.plot(rad_q, rad_I, lw=1.0, color="C3", alpha=0.8,
             label="radial average (simulated)")
    ax2.set_ylabel("Radial-average intensity / a.u. (linear)")
    ax2.set_ylim(bottom=0)

    for q, hkl in zip(reflex_q, reflex_hkl):
        if xlim[0] <= q <= xlim[1]:
            ax.axvline(q, color="grey", ls="--", lw=0.8, alpha=0.6)
            ax.annotate(
                hkl,
                xy=(q, 1.0),
                xycoords=("data", "axes fraction"),
                xytext=(0, 2),
                textcoords="offset points",
                ha="center",
                va="bottom",
                rotation=90,
                fontsize=7,
                color="grey",
            )

    handles_1, labels_1 = ax.get_legend_handles_labels()
    handles_2, labels_2 = ax2.get_legend_handles_labels()
    ax.legend(handles_1 + handles_2, labels_1 + labels_2, fontsize=8, loc="upper right")

    fig.tight_layout()
    save_figure(fig, f"{data_file.stem}_{sample_name}_Rotational_average_invAngstrom_hkl.png")
    plt.show()
else:
    print("Skipped simulation overlay. Enable plot_simulation_overlay and provide an existing simulation_xlsx_path to run it.")